# HW4 — FASTA Parser

A FASTA parser that:
- Collects file statistics via **SeqKit** (`subprocess`)
- Reads sequences via **BioPython**
- Identifies **UniProt** or **Ensembl** IDs using regex
- Queries the corresponding **REST API**
- Returns a structured nested dictionary

In [6]:
! pip install biopython
! pip install -q condacolab
!wget https://github.com/shenwei356/seqkit/releases/download/v2.8.2/seqkit_linux_amd64.tar.gz
!tar -xzf seqkit_linux_amd64.tar.gz
!mv seqkit /usr/local/bin/

--2026-04-03 00:48:59--  https://github.com/shenwei356/seqkit/releases/download/v2.8.2/seqkit_linux_amd64.tar.gz
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/52715040/59195c1d-5ade-49b8-aaff-51c23b7ff6dc?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-03T01%3A40%3A26Z&rscd=attachment%3B+filename%3Dseqkit_linux_amd64.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-03T00%3A39%3A45Z&ske=2026-04-03T01%3A40%3A26Z&sks=b&skv=2018-11-09&sig=Y7vnwPJO890qaAqMtVLFCRO5%2BzngnUqQ0G5unGPJn90%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NTE3NzU4MCwibmJmIjoxNzc1MTc3MjgwLCJwYXRoIjoicmVsZWFzZWFzc2V

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports

In [18]:
import requests
import re
import json
import sys
import subprocess
from Bio import SeqIO

## `MyFastaParser` class

In [19]:
class MyFastaParser:
    """
    FASTA parser that:
      1. Runs `seqkit stats` to get file metadata and detect sequence type.
      2. Parses the FASTA file with BioPython.
      3. Extracts UniProt / Ensembl IDs from description lines via regex.
      4. Queries the corresponding REST API and returns a structured dict.
    """

    _UNIPROT_RE = re.compile(
        r'(?:^|\|)([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]{5}|[A-NR-Z][0-9][A-Z][A-Z0-9]{2}[0-9])(?:\||\s|$)'
    )
    _ENSEMBL_RE = re.compile(
        r'(ENS[A-Z]*T\d+(?:\.\d+)?)'
    )

    def __init__(self, file_name: str):
        self.filename = file_name

    # Private utility methods — API calls

    def _get_uniprot(self, accession: str) -> requests.Response:
        """GET request to UniProt REST API."""
        url = f"https://rest.uniprot.org/uniprotkb/{accession}.json"
        response = requests.get(url)
        return response

    def _get_ensembl(self, ensembl_id: str) -> requests.Response:
        """GET request to Ensembl REST API."""
        clean_id = ensembl_id.split('.')[0]
        url = f"https://rest.ensembl.org/lookup/id/{clean_id}?content-type=application/json"
        response = requests.get(url)
        return response

    def _uniprot_parse_response(self, resp: requests.Response) -> dict:
        """Parse UniProt JSON response into a clean dict."""
        try:
            data = resp.json()
            accession = data.get("primaryAccession", "unknown")
            return {
                accession: {
                    "organism":     data.get("organism", {}).get("scientificName"),
                    "geneInfo":     data.get("genes"),
                    "sequenceInfo": data.get("sequence"),
                    "type":         "protein",
                }
            }
        except Exception as e:
            return {"error": str(e)}

    def _ensembl_parse_response(self, resp: requests.Response) -> dict:
        """Parse Ensembl JSON response into a clean dict."""
        try:
            data = resp.json()
            ens_id = data.get("id", "unknown")
            return {
                ens_id: {
                    "object_type":          data.get("object_type"),
                    "assembly_name":        data.get("assembly_name"),
                    "species":              data.get("species"),
                    "db_type":              data.get("db_type"),
                    "biotype":              data.get("biotype"),
                    "display_name":         data.get("display_name"),
                    "id":                   data.get("id"),
                    "description":          data.get("description"),
                    "canonical_transcript": data.get("canonical_transcript"),
                    "source":               data.get("source"),
                }
            }
        except Exception as e:
            return {"error": str(e)}

    def _access_database(self,
                         seq_id: str,
                         database: str,
                         seq_description: str,
                         seq_sequence: str) -> dict:
        """
        Dispatcher: calls the right API based on `database`,
        parses the response, and wraps everything into a result dict.
        """
        result = {
            f"file_info_{seq_id}": {
                "description": seq_description,
                "sequence":    seq_sequence,
            }
        }

        if database == "uniprot":
            resp = self._get_uniprot(seq_id)
            if resp.status_code == 200:
                parsed = self._uniprot_parse_response(resp)
                result[f"database_info_{seq_id}"] = parsed.get(seq_id, parsed)
            else:
                result[f"database_info_{seq_id}"] = {
                    "error": f"UniProt returned HTTP {resp.status_code} for '{seq_id}'"
                }

        elif database == "ensembl":
            resp = self._get_ensembl(seq_id)
            if resp.status_code == 200:
                parsed = self._ensembl_parse_response(resp)
                clean_id = seq_id.split('.')[0]
                result[f"database_info_{seq_id}"] = parsed.get(clean_id, parsed)
            else:
                result[f"database_info_{seq_id}"] = {
                    "error": f"Ensembl returned HTTP {resp.status_code} for '{seq_id}'"
                }
        else:
            result["WARNING"] = {"No ID match found."}

        return result

    # Public method 1 — SeqKit stats

    def seqkit_stats(self) -> dict:
        """
        Runs `seqkit stats --all --tabular` on self.filename via subprocess.

        Returns a dict with keys:
          - 'fasta_seqkit_stat_info': all columns from seqkit output
          - 'fasta_type': 'DNA' | 'Protein' | ...
          - 'fasta_num_seqs': int

        On subprocess error, returns:
          {'seqkit_error': <stderr text>}
        """
        cmd = ["seqkit", "stats", "--all", "--tabular", self.filename]

        proc = subprocess.run(
            cmd,
            capture_output=True,
            text=True
        )

        # Error branch
        if proc.returncode != 0 or proc.stderr.strip():
            error_msg = proc.stderr.strip() or f"seqkit exited with code {proc.returncode}"
            print(f"[seqkit ERROR] {error_msg}", file=sys.stderr)
            return {"seqkit_error": error_msg}

        #Parse TSV output
        lines = proc.stdout.strip().splitlines()
        if len(lines) < 2:
            return {"seqkit_error": "Unexpected seqkit output (too few lines)."}

        headers = lines[0].split('\t')
        values  = lines[1].split('\t')

        # seqkit --tabular header: file, format, type, num_seqs, ...
        # drop the 'file' column (index 0)
        stat_keys = [h.replace('(', '').replace(')', '').replace('%', '') for h in headers[1:]]
        stat_vals = values[1:]

        stat_dict = dict(zip(stat_keys, stat_vals))

        fasta_type     = stat_dict.get("type", "Unknown")
        fasta_num_seqs = int(stat_dict.get("num_seqs", 0))

        return {
            "fasta_seqkit_stat_info": stat_dict,
            "fasta_type":             fasta_type,
            "fasta_num_seqs":         fasta_num_seqs,
        }


    #Public method 2 — BioPython parser

    def biopython_parser(self, seqkit_result: dict) -> dict:
        """
        Accepts the dict returned by seqkit_stats().

        - If seqkit reported an error, returns that error unchanged.
        - Otherwise reads the FASTA file with BioPython, identifies IDs
          via regex appropriate for the sequence type, queries the DB,
          and returns a combined result dict.
        """

        #Propagate seqkit error
        if "seqkit_error" in seqkit_result:
            return seqkit_result

        fasta_type = seqkit_result.get("fasta_type", "")

        #Choose regex and database based on sequence type
        # Protein  → UniProt
        # DNA/RNA  → Ensembl
        if fasta_type == "Protein":
            id_regex = self._UNIPROT_RE
            database = "uniprot"
        else:
            id_regex = self._ENSEMBL_RE
            database = "ensembl"

        output = {"DB_name": database}

        try:
            records = list(SeqIO.parse(self.filename, "fasta"))
        except Exception as e:
            return {"biopython_error": str(e)}

        for record in records:
            description = record.description
            sequence    = str(record.seq)

            match = id_regex.search(description)

            if match:
                seq_id = match.group(1)
                db_result = self._access_database(
                    seq_id      = seq_id,
                    database    = database,
                    seq_description = description,
                    seq_sequence    = sequence,
                )
            else:
                seq_id = record.id
                db_result = {
                    f"file_info_{seq_id}": {
                        "description": description,
                        "sequence":    sequence,
                    },
                    "WARNING": {"No ID match found."},
                }

            output.update(db_result)

        return output

    # Public method 3 — Pretty-print

    def show_output(self, output: dict, indent: int = 0):
        """Recursively pretty-print the result dict."""
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))


## Testing


### Test 1 — `test_file.fasta` (Protein / UniProt + unknown PDB entry)

In [20]:
parser = MyFastaParser('/content/drive/MyDrive/test_file.fasta')
stats = parser.seqkit_stats()
stats

{'fasta_seqkit_stat_info': {'format': 'FASTA',
  'type': 'Protein',
  'num_seqs': '2',
  'sum_len': '456',
  'min_len': '29',
  'avg_len': '228.0',
  'max_len': '427',
  'Q1': '29.0',
  'Q2': '228.0',
  'Q3': '427.0',
  'sum_gap': '0',
  'N50': '427',
  'N50_num': '1',
  'Q20': '0.00',
  'Q30': '0.00',
  'AvgQual': '0.00',
  'GC': '8.33'},
 'fasta_type': 'Protein',
 'fasta_num_seqs': 2}

In [21]:
biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
	organism
		Homo sapiens
	geneInfo
		[{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
	sequenceInfo
		value
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSS

### Test 2 — `uniprot_download.fasta` (Protein / UniProt)

In [22]:
parser2 = MyFastaParser('/content/drive/MyDrive/uniprot_download.fasta')
stats2  = parser2.seqkit_stats()
stats2

{'fasta_seqkit_stat_info': {'format': 'FASTA',
  'type': 'Protein',
  'num_seqs': '7',
  'sum_len': '3861',
  'min_len': '180',
  'avg_len': '551.6',
  'max_len': '1382',
  'Q1': '429.0',
  'Q2': '441.0',
  'Q3': '500.0',
  'sum_gap': '0',
  'N50': '468',
  'N50_num': '3',
  'Q20': '0.00',
  'Q30': '0.00',
  'AvgQual': '0.00',
  'GC': '9.14'},
 'fasta_type': 'Protein',
 'fasta_num_seqs': 7}

In [23]:
biopython2 = parser2.biopython_parser(stats2)
parser2.show_output(biopython2)

DB_name
	uniprot
file_info_Q9R1A7
	description
		sp|Q9R1A7|NR1I2_RAT Nuclear receptor subfamily 1 group I member 2 OS=Rattus norvegicus OX=10116 GN=Nr1i2 PE=2 SV=1
	sequence
		MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLLPHLADVSTYMFKGVINFAKVISHFRELPIEDQISLLKGATFEMCILRFNTMFDTETGTWECGRLAYCFEDPNGGFQKLLLDPLMKFHCMLKKLQLREEEYVLMQAISLFSPDRPGVVQRSVVDQLQERFALTLKAYIECSRPYPAHRFLFLKIMAVLTELRSINAQQTQQLLRIQDTHPFATPLMQELFSSTDG
database_info_Q9R1A7
	organism
		Rattus norvegicus
	geneInfo
		[{'geneName': {'value': 'Nr1i2'}, 'synonyms': [{'value': 'Pxr'}]}]
	sequenceInfo
		value
			MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLL

### Test 3 — `ensembl_download_1.fasta` (DNA / Ensembl — correct file)

In [24]:
parser3 = MyFastaParser('/content/drive/MyDrive/ensembl_download_1.fasta')
stats3  = parser3.seqkit_stats()
stats3

{'fasta_seqkit_stat_info': {'format': 'FASTA',
  'type': 'DNA',
  'num_seqs': '6',
  'sum_len': '86',
  'min_len': '9',
  'avg_len': '14.3',
  'max_len': '23',
  'Q1': '10.0',
  'Q2': '13.5',
  'Q3': '17.0',
  'sum_gap': '0',
  'N50': '16',
  'N50_num': '3',
  'Q20': '0.00',
  'Q30': '0.00',
  'AvgQual': '0.00',
  'GC': '45.35'},
 'fasta_type': 'DNA',
 'fasta_num_seqs': 6}

In [25]:
biopython3 = parser3.biopython_parser(stats3)
parser3.show_output(biopython3)

DB_name
	ensembl
file_info_ENSMUST00000196221.2
	description
		ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
	sequence
		ATGGCATAT
database_info_ENSMUST00000196221.2
	object_type
		Transcript
	assembly_name
		GRCm39
	species
		mus_musculus
	db_type
		core
	biotype
		TR_D_gene
	display_name
		Trdd1-202
	id
		ENSMUST00000196221
	description
		None
	canonical_transcript
		None
	source
		havana
file_info_ENSMUST00000177564.2
	description
		ENSMUST00000177564.2 cds chromosome:GRCm39:14:54359683:54359698:1 gene:ENSMUSG00000096176.2 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd2 description:T cell receptor delta diversity 2 [Source:MGI Symbol;Acc:MGI:4439546]
	sequence
		ATCGGAGGGATACGAG
database_info_ENSMUST00000177564.2
	object_type
		Transcript
	assembly_name
		GRCm39
	spe

### Test 4 — `ensembl_download_2.fasta` (invalid — first record missing `>`)

In [28]:
parser4 = MyFastaParser('/content/drive/MyDrive/ensembl_download_2.fasta')
stats4  = parser4.seqkit_stats()
stats4

[seqkit ERROR] [ERRO] /content/drive/MyDrive/ensembl_download_2.fasta: fastx: invalid FASTA/Q format


{'seqkit_error': '\x1b[ERRO]\x1b /content/drive/MyDrive/ensembl_download_2.fasta: fastx: invalid FASTA/Q format'}

In [29]:
biopython4 = parser4.biopython_parser(stats4)
parser4.show_output(biopython4)

seqkit_error
	[ERRO] /content/drive/MyDrive/ensembl_download_2.fasta: fastx: invalid FASTA/Q format
